In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__notebook__.ipynb
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__output__.json
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__results__.html
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/custom.css
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_test.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_train.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_val.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/y_test.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/y_train.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/y_val.csv


In [2]:
from pathlib import Path

DATA_DIR = Path("/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")

y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()
y_val = pd.read_csv(DATA_DIR / "y_val.csv").squeeze()
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze()

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(28831, 19)
(6178, 19)
(6179, 19)
(28831,)
(6178,)
(6179,)


In [3]:
v2_drop=["nr.employed", "euribor3m"]
X_train.drop(columns=v2_drop, inplace=True)
X_val.drop(columns=v2_drop, inplace=True)
X_test.drop(columns=v2_drop, inplace=True)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(28831, 17)
(6178, 17)
(6179, 17)


In [4]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Identify numerical and categorical columns
cat_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

num_cols = X_train.select_dtypes(
    include=np.number
).columns.tolist()

# Numerical preprocessing
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical preprocessing
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        drop='first'
    ))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Fit on train; transform validation and test
X_train_encoded = preprocessor.fit_transform(X_train)
X_val_encoded = preprocessor.transform(X_val)
X_test_encoded = preprocessor.transform(X_test)

print("Train:", X_train_encoded.shape)
print("Validation:", X_val_encoded.shape)
print("Test:", X_test_encoded.shape)

Train: (28831, 50)
Validation: (6178, 50)
Test: (6179, 50)


In [5]:
from xgboost import XGBClassifier

# xgb = XGBClassifier(
#     n_estimators=200,
#     learning_rate=0.1,
#     max_depth=6,
#     random_state=42,
#     eval_metric='logloss',
#     n_jobs=-1
# )

In [6]:
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [7]:
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 500
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 3, 10
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 10
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),

        "gamma": trial.suggest_float(
            "gamma", 0, 5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 10,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 10,
            log=True
        )
    }

    model = XGBClassifier(
        **params,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
        tree_method="hist"
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    return scores.mean()

In [8]:
os.makedirs("/kaggle/working/outputs", exist_ok=True)

log_path = "/kaggle/working/outputs/optuna_xgb_trials.csv"

def log_optuna_trial(study, trial):

    rows = []

    for t in study.trials:

        row = {
            "Trial": t.number,
            "Current CV ROC-AUC": t.value,
            "Best CV ROC-AUC": study.best_value,
            "Best Trial": study.best_trial.number,
            "Is Best": t.number == study.best_trial.number
        }

        row.update(t.params)
        rows.append(row)

    pd.DataFrame(rows).sort_values("Trial").to_csv(
        log_path,
        index=False
    )

In [9]:
!pip install -q optuna
import optuna

In [10]:
study = optuna.create_study(
    direction="maximize",
    study_name="xgboost_roc_auc"
)

study.optimize(
    objective,
    n_trials=150,
    callbacks=[log_optuna_trial]
)

[I 2026-11-20 08:38:35,802] A new study created in memory with name: xgboost_roc_auc
[I 2026-11-20 08:38:39,942] Trial 0 finished with value: 0.7894814542857512 and parameters: {'n_estimators': 182, 'learning_rate': 0.011171319964639177, 'max_depth': 4, 'min_child_weight': 5, 'subsample': 0.8830530160636909, 'colsample_bytree': 0.6747758646724484, 'gamma': 3.4655100944909862, 'reg_alpha': 1.0034596922086335e-08, 'reg_lambda': 6.04949699024184}. Best is trial 0 with value: 0.7894814542857512.
[I 2026-11-20 08:38:41,383] Trial 1 finished with value: 0.788960029474367 and parameters: {'n_estimators': 139, 'learning_rate': 0.19514796318044347, 'max_depth': 4, 'min_child_weight': 6, 'subsample': 0.7927263598881722, 'colsample_bytree': 0.937094261133193, 'gamma': 1.1610199613291754, 'reg_alpha': 2.437510824637536, 'reg_lambda': 0.008568568080600397}. Best is trial 0 with value: 0.7894814542857512.
[I 2026-11-20 08:38:42,675] Trial 2 finished with value: 0.7942223194825256 and parameters: {'n

In [11]:
optuna_results = pd.read_csv(
    "/kaggle/working/outputs/optuna_xgb_trials.csv"
)

optuna_results.round(4)

,Trial,Current CV ROC-AUC,Best CV ROC-AUC,Best Trial,Is Best,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda
0,0,0.7895,0.7953,126,False,182,0.0112,4,5,0.8831,0.6748,3.4655,0.0000,6.0495
1,1,0.7890,0.7953,126,False,139,0.1951,4,6,0.7927,0.9371,1.1610,2.4375,0.0086
2,2,0.7942,0.7953,126,False,174,0.1542,7,7,0.9607,0.6199,3.1527,0.0000,0.0073
3,3,0.7926,0.7953,126,False,366,0.0300,6,4,0.9947,0.7552,1.0350,0.0000,0.1206
4,4,0.7923,0.7953,126,False,204,0.0150,9,9,0.7767,0.7698,1.3591,0.1307,0.0027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,145,0.7940,0.7953,126,False,126,0.1162,8,6,0.9852,0.6367,3.6212,0.0000,0.0013
146,146,0.7940,0.7953,126,False,153,0.1444,10,7,0.9839,0.6250,3.7451,0.0000,0.0033
147,147,0.7934,0.7953,126,False,114,0.1203,8,6,0.9538,0.6451,3.7564,0.0000,0.0015
148,148,0.7938,0.7953,126,False,136,0.1009,9,6,0.9472,0.6137,3.1769,0.0000,0.0013


In [12]:
print("Best Trial:", study.best_trial.number)
print("Best CV ROC-AUC:", study.best_value)
print("Best Parameters:")

for param, value in study.best_params.items():
    print(f"{param}: {value}")

Best Trial: 126
Best CV ROC-AUC: 0.795265556577254
Best Parameters:
n_estimators: 141
learning_rate: 0.1534343642146888
max_depth: 10
min_child_weight: 7
subsample: 0.969316156131328
colsample_bytree: 0.6548090293428045
gamma: 3.628161066232197
reg_alpha: 2.4530090351206983e-06
reg_lambda: 0.0030405818856991127


In [13]:
best_xgb = XGBClassifier(
    **study.best_params,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
    tree_method="hist"
)

best_xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_xgb)
])

best_xgb_pipeline.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['age', 'campaign', 'pdays',
                                                   'previous', 'emp.var.rate',
                                                   'cons.price.idx',
                                                   'cons.conf.idx']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='i...
                               gamma=3.628161066232197, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.1534343642146888, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=10,
                               max_leaves=None, min_child_weight=7, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=141, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

y_val_pred = best_xgb_pipeline.predict(X_val)
y_val_prob = best_xgb_pipeline.predict_proba(X_val)[:, 1]

tuned_xgb_results = {
    "Model": "XGBoost Tuned",
    "Accuracy": accuracy_score(y_val, y_val_pred),
    "Precision": precision_score(y_val, y_val_pred),
    "Recall": recall_score(y_val, y_val_pred),
    "F1": f1_score(y_val, y_val_pred),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob)
}

pd.DataFrame([tuned_xgb_results]).round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,XGBoost Tuned,0.9011,0.6628,0.2486,0.3615,0.8056


In [15]:
# models_cv = {
#     "XGBoost": (xgb, X_train_encoded)
# }

# cv_results = []

# for name, (model, X_train_model) in models_cv.items():

#     scores = cross_validate(
#         model,
#         X_train_model,
#         y_train,
#         cv=cv,
#         scoring=[
#             "roc_auc",
#             "precision",
#             "recall",
#             "f1"
#         ],
#         n_jobs=-1
#     )

#     cv_results.append({
#         "Model": name,
#         "CV ROC-AUC": scores["test_roc_auc"].mean(),
#         "CV Precision": scores["test_precision"].mean(),
#         "CV Recall": scores["test_recall"].mean(),
#         "CV F1": scores["test_f1"].mean()
#     })

# cv_results_df = pd.DataFrame(cv_results)

# cv_results_df.round(4)

In [16]:
# xgb.fit(X_train_encoded, y_train)

In [17]:
# xgb_prob = xgb.predict_proba(X_val_encoded)[:, 1]

In [18]:
# from sklearn.metrics import (
#     accuracy_score,
#     precision_score,
#     recall_score,
#     f1_score,
#     roc_auc_score
# )
# models = {
#     "XGBoost": (xgb, X_val_encoded)
# }

# results = []

# for name, (model, X_val_model) in models.items():

#     y_pred = model.predict(X_val_model)
#     y_prob = model.predict_proba(X_val_model)[:, 1]

#     results.append({
#         "Model": name,
#         "Accuracy": accuracy_score(y_val, y_pred),
#         "Precision": precision_score(y_val, y_pred),
#         "Recall": recall_score(y_val, y_pred),
#         "F1": f1_score(y_val, y_pred),
#         "ROC-AUC": roc_auc_score(y_val, y_prob)
#     })